# Ejemplos con Hive

## Descripción de las variables

El dataset, obtenido de <a target = "_blank" href="https://www.transtats.bts.gov/Fields.asp?Table_ID=236">este link</a> está compuesto por las siguientes variables referidas siempre al año 2018:

1. **Month** 1-4
2. **DayofMonth** 1-31
3. **DayOfWeek** 1 (Monday) - 7 (Sunday)
4. **FlightDate** fecha del vuelo
5. **Origin** código IATA del aeropuerto de origen
6. **OriginCity** ciudad donde está el aeropuerto de origen
7. **Dest** código IATA del aeropuerto de destino
8. **DestCity** ciudad donde está el aeropuerto de destino  
9. **DepTime** hora real de salida (local, hhmm)
10. **DepDelay** retraso a la salida, en minutos
11. **ArrTime** hora real de llegada (local, hhmm)
12. **ArrDelay** retraso a la llegada, en minutos: se considera que un vuelo ha llegado "on time" si aterrizó menos de 15 minutos más tarde de la hora prevista en el Computerized Reservations Systems (CRS).
13. **Cancelled** si el vuelo fue cancelado (1 = sí, 0 = no)
14. **CancellationCode** razón de cancelación (A = aparato, B = tiempo atmosférico, C = NAS, D = seguridad)
15. **Diverted** si el vuelo ha sido desviado (1 = sí, 0 = no)
16. **ActualElapsedTime** tiempo real invertido en el vuelo
17. **AirTime** en minutos
18. **Distance** en millas
19. **CarrierDelay** en minutos: El retraso del transportista está bajo el control del transportista aéreo. Ejemplos de sucesos que pueden determinar el retraso del transportista son: limpieza de la aeronave, daño de la aeronave, espera de la llegada de los pasajeros o la tripulación de conexión, equipaje, impacto de un pájaro, carga de equipaje, servicio de comidas, computadora, equipo del transportista, problemas legales de la tripulación (descanso del piloto o acompañante) , daños por mercancías peligrosas, inspección de ingeniería, abastecimiento de combustible, pasajeros discapacitados, tripulación retrasada, servicio de inodoros, mantenimiento, ventas excesivas, servicio de agua potable, denegación de viaje a pasajeros en mal estado, proceso de embarque muy lento, equipaje de mano no válido, retrasos de peso y equilibrio.
20. **WeatherDelay** en minutos: causado por condiciones atmosféricas extremas o peligrosas, previstas o que se han manifestado antes del despegue, durante el viaje, o a la llegada.
21. **NASDelay** en minutos: retraso causado por el National Airspace System (NAS) por motivos como condiciones meteorológicas (perjudiciales pero no extremas), operaciones del aeropuerto, mucho tráfico aéreo, problemas con los controladores aéreos, etc.
22. **SecurityDelay** en minutos: causado por la evacuación de una terminal, re-embarque de un avión debido a brechas en la seguridad, fallos en dispositivos del control de seguridad, colas demasiado largas en el control de seguridad, etc.
23. **LateAircraftDelay** en minutos: debido al propio retraso del avión al llegar, problemas para conseguir aterrizar en un aeropuerto a una hora más tardía de la que estaba prevista.

Leemos el fichero CSV utilizando el delimitador por defecto de Spark (","). La primera línea contiene encabezados (nombres de columnas) por lo que no es parte de los datos y debemos indicarlo con la opción header.

## 1. Leemos los datos del fichero CSV que hay en Azure Data Lake Storage

In [0]:
# Esto no hace nada: la lectura es lazy así que no se lee en realidad hasta que ejecutemos una acción sobre flightsDF
# Solamente se comprueba que exista el fichero en esa ruta, y se leen los nombres de columnas
flightsDF = spark.read.option("header", "true")\
                 .option("inferSchema", "true")\
                 .csv("/Volumes/workspace/default/datos/flights-jan-apr-2018.csv")

## 2. Vamos a mostrar el contenido del metastore de Hive, que ahora mismo no tiene nada

In [0]:
spark.sql("select current_catalog()").show()

spark.sql("show tables").show()

## 3. Creamos una vista temporal de un DF que tiene solo dos columnas

Esto solamente añade metadatos al metastore de Hive, que además se borrarán cuando cerremos el notebook. No guarda datos físicos de la tabla en ningún lado, puesto que el DF está en memoria. O mejor dicho, el DF del que proviene esta vista temporal "no está en ningún lado", porque no hemos cacheado weatherDistanceDF así que cualquier consulta SQL que hagamos sobre la tabla `weatherDistanceTable` provoca que se tenga que re-calcular el DF `weatherDistanceDF` sobre el cual está creada dicha tabla temporal.

In [0]:
weatherDistanceDF = flightsDF.select("WeatherDelay", "Distance")
weatherDistanceDF.createOrReplaceTempView("weatherDistanceTable")

Ahora vemos que la tabla se ha creado como vista temporal. Registrar un DF como vista temporal no requiere en absoluto que el DF esté materializado.

In [0]:
spark.sql("show tables").show()

Podemos hacer consultas sobre ella

In [0]:
flightsLejosDF = spark.sql("select * from weatherDistanceTable where Distance > 200 and WeatherDelay is not null limit 5")
flightsLejosDF.show()

Cacheamos la tabla con una operación que **sí la materializa inmediatamente**. El comando CACHE TABLE es similar a cache() pero sí actúa de manera inmediata.

In [0]:
# spark.sql("CACHE TABLE weatherDistanceTable")

## 4. Creamos una tabla persistente manejada, guardando como tabla el contenido de un DF

Primero vamos a crear una database específica que no sea la default, que es la única database pre-existente.

In [0]:
spark.sql("create database if not exists workspace.vuelos")

In [0]:
flightsJFK = flightsDF.where("Origin = 'JFK'") #.cache()
flightsJFK.write.saveAsTable("workspace.vuelos.flightsjfk") # es una acción: se guardan físicamente los datos en una ubicación de DBFS configurada para las tablas manejads

In [0]:
# Databricks Free Edition ya no almacena las tablas manejadas en DBFS sino en su almacenamiento interno por defecto,
# que probablemente está soportado por un bucket de Amazon S3
# !ls -l /dbfs/user/hive/warehouse/vuelos.db/flightsjfk/

Si volvemos a mostrar las tablas que existen, veremos la nueva. Vemos que **no** es temporal, pero no sabemos si es manejada o es externa.

In [0]:
spark.sql("show tables in vuelos").display()  # muestra las tablas de la database vuelos. Podemos omitir esto si fijamos la database vuelos como la activa

In [0]:
#spark.sql("use database vuelos")
spark.sql("show tables").display()   # de aquí en adelante no será necesario indicar la database

In [0]:
spark.sql("describe extended workspace.vuelos.flightsjfk").display()

* Como vemos, el campo Location indica la carpeta donde se están guardando las tablas *manejadas*, que es la carpeta /user/hive/warehouse/<nombretabla> de DBFS. 
* Por defecto el formato de fichero en Databricks es Delta

## 5. Guardamos flightsDF como fichero Delta en ADLS, sin crear ninguna tabla en el metastore

Lo vamos a guardar particionado por la columna Origin. Como este DF sólo tiene dos aeropuertos distintos porque hemos retenido solamente los vuelos que salen de SFO o de LAX, Spark creará dos subcarpetas. Dentro de cada subcarpeta habrá tantos ficheros como particiones tiene el DF, que actualmente son 3. Se puede comprobar con `flightsSFO.rdd.getNumPartitions()`

In [0]:
flightsSFO = flightsDF.where("Origin = 'SFO' or Origin = 'LAX'")\
                      .select("FlightDate", "Origin", "Dest", "Distance")

flightsSFO.write.mode("overwrite")\
                .partitionBy("Origin")\
                .format("delta")\
                .save("/Volumes/workspace/default/datos/flightsSFO")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/datos/flightsSFO"))

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/datos/flightsSFO/Origin=LAX"))

In [0]:
# Databricks Free Edition no permite el acceso al RDD interno de un DataFrame (el cluster serverless no lo permite)
# flightsSFO.rdd.getNumPartitions()  

In [0]:
read_sfo_df = spark.read.format("delta")\
                        .load("/Volumes/workspace/default/datos/flightsSFO")
read_sfo_df.limit(10).display()

## 6. Ahora vamos a crear una tabla EXTERNA a partir del fichero existente /flightsSFO (que en realidad es una carpeta)

#### Databricks Free Edition, al traer Unity Catalog activado, no permite crear tablas externas sin primero crear una External Location. Sin embargo, este proceso es tedioso porque requiere crear primero una credencial administrada, y en Free Edition nos obligaría a hacerlo apuntando a un bucket de Amazon Web Services (no se puede hacer apuntando al contenedor de Azure por restricciones en la identificación). Además, DBFS está ya desactivado en Free Edition

## Vamos a comprobar el efecto de guardar un DF en una carpeta ya existente, sobreescribiendo solamente las particiones de los valores que estén presentes en los datos que ahora estamos guardando

Como flightsJFK sólo tiene Origin el aeropuerto JFK, podríamos guardarlo en la misma ubicación que flightsSFO y pedirle `.mode("overwrite").partitionBy("Origin")` y esa operación NO borrará lo que ya había sino que solamente reemplazará las particiones que ya existieran. Como la única partición que ahora se va a crear es la de JFK y esa no existía, el resultado es que simplemente se añade una partición más (es decir una subcarpeta Origin=JFK) a la carpeta.

**IMPORTANTE**: el DataFrame que vamos a guardar particionado para sobreescribir ciertas particiones debe tener exactamente EL MISMO ESQUEMA que los datos de las particiones ya existentes.

**IMPORTANTE**: esta operación funciona cuando el parámetro de Spark llamado `"spark.sql.sources.partitionOverwriteMethod"` está fijado al valor `"dynamic"`, o bien cuando indicamos en la propia operación de escritura dicho parámetro con la opción `option("partitionOverwriteMode", "dynamic")`.

El método habitual para dar valor a los parámetros es llamar a `spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")`. Podemos consultar qué valor tiene actualmente con
`spark.conf.get("spark.sql.sources.partitionOverwriteMode")` y veremos que está configurado como `STATIC`.

Aunque en este ejemplo en realidad hemos "añadido" una nueva partición porque los datos que vamos a guardar la segunda vez no contienen ningún origen ya existente (solamente un Origin *nuevo* que es JFK), ocurriría exactamente lo mismo si los datos sí tuviesen algún aeropuerto de particiones ya existentes: se reemplazaría completa la partición de ese aeropuerto ya existente. Si además hubiese aeropuertos que todavía no existían y no tenían subcarpeta, se crearían nuevas particiones (nuevas subcarpetas) para dichos aeropuertos, sin tocar las subcarpetas existentes.

In [0]:
# IMPRESCINDIBLE para que sólo se reemplace en la carpeta las particiones que estemos escribiendo (recordar poner partitionBy porque en caso contrario se reemplaza la carpeta completa!)

# Esta propiedad ya viene configurada por defecto como "dynamic" en Free Edition
#spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

# IMPRESCINDIBLE para que el DataFrame que vamos a guardar tenga el mismo esquema que los datos ya existentes
flightsJFK.select("FlightDate", "Origin", "Dest", "Distance")\
          .write\
          .mode("overwrite")\
          .partitionBy("Origin")\
          .format("delta")\
          .save("/Volumes/workspace/default/datos/flightsSFO")

### Comprobamos que ha añadido una subcarpeta `Origin=JFK` y no ha borrado nada de lo que había en esa carpeta

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/datos/flightsSFO"))

### Probamos la opción `replaceWhere` específico del formato Delta

Permite reemplazar filas de acuerdo a cierta condición, tanto si la condición se refiere a columnas por las cuales está particionada la carpeta como si hace referencia a otras columnas.

In [0]:
from datetime import date

nuevo_df = spark.createDataFrame(
    [(date(2024, 2, 1), "LAX", "GRX", 6000.0),
     (date(2024, 2, 1), "SFO", "GRX", 5500.0),
     (date(2024, 2, 1), "LAS", "GRX", 6200.0)],
    schema="FlightDate date, Origin string, Dest string, Distance double")

In [0]:
spark.read.format("delta").load("/Volumes/workspace/default/datos/flightsSFO").printSchema()

**La siguiente escritura fallará:** Para poder escribir un DF que contiene datos con más valores que los indicados en replaceWhere, hay que indicar que no valide la constraint que por defecto comprueba, que consiste en que el DF debe contener exclusivamente filas cuyo valor esté contemplado en la condición de replaceWhere

In [0]:
# No necesitamos indicarle cómo está particionada la carpeta donde estamos escribiendo

nuevo_df.write\
        .mode("overwrite")\
        .option("replaceWhere", "Origin = 'SFO'")\
        .format("delta")\
        .save("/Volumes/workspace/default/datos/flightsSFO")

**Desactivamos la constraint y probamos de nuevo la escritura**

* **IMPORTANTE**: no podemos especificar la opción ad-hoc `option("partitionOverwriteMode", "dynamic")` y la opción `option("replaceWhere", "...")` en una misma operación de escritura. 
* Sí está permitido utilizar replaceWhere cuando el particionamiento dinámico está configurado como parámetro de Spark, puesto que dicho parámetro será ignorado, y el comportamiento lo marcará replaceWhere

In [0]:
# Databricks Free Edition no permite fijar propiedades de Spark y tampoco podemos desactivarla como option en la escritura

#spark.conf.set("spark.databricks.delta.replaceWhere.constraintCheck.enabled", False)  # desactivar la restricción anterior

# Por tanto no es posible escribir los nuevos datos si no los hemos filtrado nosotros manualmente, con lo que ya no podremos añadir además las filas nuevas tal como pretendíamos (las habremos tenido que quitar antes de escribir)

(nuevo_df
        .where("Origin = 'SFO'")  # estamos obligados a hacer esto para que se cumpla la CONSTRAINT de replaceWhere
        .write
        .mode("overwrite")
        .option("replaceWhere", "Origin = 'SFO'")
        .format("delta")
        .save("/Volumes/workspace/default/datos/flightsSFO")
)

### Comprobamos qué ha provocado el guardado anterior

El reemplazamiento estaba indicado solamente para las filas de SFO. Debería haber ocurrido lo siguiente:
* Debe haberse añadido (a pesar de que no fue necesario especificar particionamiento) una nueva subcarpeta (una nueva partición) para el nuevo origen LAS que no existía. 
  * **NO OCURRE ASÍ EN DATABRICKS FREE EDITION PORQUE LO HEMOS ELIMINADO EN EL FILTRADO**
* Debería haberse añadido la fila de la partición existente LAX, puesto que dicho valor no se incluyó en el replaceWhere pero como hay un dato, debe añadirse. 
  * **NO OCURRE ASÍ EN DATABRICKS FREE EDITION PORQUE LO HEMOS ELIMINADO EN EL FILTRADO** -->
* Debería haberse reemplazado el contenido de toda la subcarpeta de SFO por una única fila, el vuelo con Origin=SFO y Dest=GRX, y por tanto habrá desaparecido todo lo que había en esa partición y ahora habrá una sola fila.

Consultamos tanto la estructura de carpetas como el contenido del DF que podemos leer.

In [0]:
flights_read = spark.read.format("delta").load("/Volumes/workspace/default/datos/flightsSFO")
# también podríamos: flights_read = spark.read.format("delta").load("abfss://datos@cursosparkucm.dfs.core.windows.net/flightsSFO")
flights_read.where("FlightDate = '2024-02-01'").display()

In [0]:
flights_read.where("Origin = 'LAX'").sort("FlightDate", ascending=False).limit(5).display()
flights_read.where("Origin = 'SFO'").sort("FlightDate", ascending=False).limit(5).display()

## 7. Borramos tabla externa y comprobamos que la carpeta sigue ahí
   --> NO PODEMOS BORRAR TABLAS EXTERNAS PORQUE NO SE PUDIERON CREAR EN FREE EDITION

In [0]:
spark.sql("show tables in vuelos").show()

In [0]:
#spark.sql("drop table vuelos.flightssfo")
spark.sql("show tables in vuelos").show()

### Comprobamos que sigue existiendo la carpeta /flightsSFO

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/datos/flightsSFO"))

## 8. Comprobamos dónde están los datos de la tabla persistente manejada que habíamos guardado con saveAsTable

In [0]:
# No es posible porque no se revela el lugar donde se almacenan las tablas manejadas en Free Edition

#!ls -l /dbfs/user/hive/warehouse/vuelos.db

In [0]:
# !ls -l /dbfs/user/hive/warehouse/vuelos.db/flightsjfk

## 9. Borramos la tabla manejada y comprobamos que, al ser manejada, Spark ha borrado físicamente esos datos al borrar la tabla

In [0]:
spark.sql("drop table vuelos.flightsjfk")
spark.sql("show tables in vuelos").show()